In [31]:
import duckdb

print(duckdb.__version__)

1.5.5


In [32]:
duckdb.sql("""
SELECT 42 AS answer
""").show()


┌────────┐
│ answer │
│ int32  │
├────────┤
│     42 │
└────────┘



In [33]:
from pathlib import Path

path = Path("data/raw/development/LCL-June2015v2_0.csv")

print(path.exists())
print(path.absolute())

False
c:\Users\welcome\Desktop\Arora\GridWise AI\notebooks\data\raw\development\LCL-June2015v2_0.csv


In [34]:
con = duckdb.connect()
con.execute("""
    CREATE OR REPLACE VIEW lcl_raw AS
    SELECT *
    FROM read_csv_auto('../data/raw/development/LCL-June2015v2_0.csv')
""")

In [35]:
con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM lcl_raw
""").fetchdf()

,total_rows
0,1000000


In [36]:
import duckdb

con = duckdb.connect()

result = con.execute("""
    SELECT 42 AS answer
""").fetchall()

print(result)

[(42,)]


In [37]:
import duckdb

csv_path = "../data/raw/development/LCL-June2015v2_0.csv"

con = duckdb.connect()

result = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path}')
    LIMIT 5
""").fetchall()

print(result)


[('MAC000002', 'Std', datetime.datetime(2012, 10, 12, 0, 30), ' 0 '), ('MAC000002', 'Std', datetime.datetime(2012, 10, 12, 1, 0), ' 0 '), ('MAC000002', 'Std', datetime.datetime(2012, 10, 12, 1, 30), ' 0 '), ('MAC000002', 'Std', datetime.datetime(2012, 10, 12, 2, 0), ' 0 '), ('MAC000002', 'Std', datetime.datetime(2012, 10, 12, 2, 30), ' 0 ')]


In [38]:
result=con.execute("""
DESCRIBE 
SELECT* 
FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
""").fetchdf()

print(result)


              column_name column_type null   key default extra
0                   LCLid     VARCHAR  YES  None    None  None
1                stdorToU     VARCHAR  YES  None    None  None
2                DateTime   TIMESTAMP  YES  None    None  None
3  KWH/hh (per half hour)     VARCHAR  YES  None    None  None


1. How many rows?
2. How many households?
3. What time period?
4. Is consumption actually numeric?

In [39]:
con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
""").fetchdf()

,total_rows
0,1000000


In [40]:
con.execute("""
    SELECT COUNT(DISTINCT LCLid) AS households
    FROM read_csv_auto('../data/raw/development/LCL-June2015v2_0.csv')

""").fetchdf()

,households
0,30


In [41]:
con.execute("""
    SELECT 
        MIN(DateTime) AS earliest_reading,
        MAX(DateTime) AS latest_reading
    FROM read_csv_auto(
    '../data/raw/development/LCL-June2015v2_0.csv'
    )
    """).fetchdf()

,earliest_reading,latest_reading
0,2011-12-06 13:00:00,2014-02-28


In [42]:
con.execute("""
    SELECT
        "KWH/hh (per half hour)" AS consumption,
        COUNT (*) AS frequency
    FROM read_csv_auto(
    '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY consumption
    ORDER BY frequency DESC
    LIMIT 20
""").fetchdf()

,consumption,frequency
0,0,45538
1,0.054,7487
2,0.055,7195
3,0.056,6478
4,0.053,6289
5,0.057,6046
6,0.047,5956
7,0.052,5707
8,0.058,5459
9,0.05,5439


Are there non-numeric consumption values?


In [43]:
result_non_numeric = con.execute("""
    SELECT
        "KWH/hh (per half hour)" AS consumption
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    WHERE TRY_CAST(
        TRIM("KWH/hh (per half hour)") AS DOUBLE  
    ) IS NULL
    LIMIT 20
""").fetchdf()

print(result_non_numeric)
# Try converting this to a number. If it can't be converted, don't crash—give me NULL
# Raw consumption
#       ↓
# Remove spaces → TRIM
#       ↓
# Try converting → TRY_CAST(... AS DOUBLE)
#       ↓
# Failed conversion?
#       ↓
# YES → show it
# this is data-quality test, not our final cleaning operation.

   consumption
0         Null
1         Null
2         Null
3         Null
4         Null
5         Null
6         Null
7         Null
8         Null
9         Null
10        Null
11        Null
12        Null
13        Null
14        Null
15        Null
16        Null
17        Null
18        Null
19        Null


In [44]:
# checking Missing values
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(LCLid) AS household_id_present,

        COUNT( stdorToU) AS tariff_present,
        COUNT(DateTime) AS datetime_present,
        COUNT("KWH/hh (per half hour)") AS consumption_present
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
""").fetchdf()

,total_rows,household_id_present,tariff_present,datetime_present,consumption_present
0,1000000,1000000,1000000,1000000,1000000


Duplicate readings

In [45]:
duplicate_details = con.execute("""
    SELECT
        LCLid,
        DateTime,
        COUNT(*) AS occurrences,
                COUNT(DISTINCT stdorToU) ,
        COUNT(DISTINCT "KWH/hh (per half hour)") AS distinct_consumption_values
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY LCLid, DateTime
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 20
""").fetchdf()

print(duplicate_details)

        LCLid   DateTime  occurrences  count(DISTINCT stdorToU)  \
0   MAC000035 2012-12-21            2                         1   
1   MAC000035 2013-01-21            2                         1   
2   MAC000035 2014-02-28            2                         1   
3   MAC000010 2014-01-28            2                         1   
4   MAC000011 2013-03-24            2                         1   
5   MAC000011 2013-10-27            2                         1   
6   MAC000011 2013-11-27            2                         1   
7   MAC000012 2013-06-25            2                         1   
8   MAC000028 2013-06-25            2                         1   
9   MAC000034 2013-03-24            2                         1   
10  MAC000034 2013-10-27            2                         1   
11  MAC000034 2013-11-27            2                         1   
12  MAC000018 2011-12-15            2                         1   
13  MAC000022 2012-02-15            2                         

In [46]:
non_numeric = con.execute("""
    SELECT
        "KWH/hh (per half hour)" AS consumption
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    WHERE TRY_CAST(
        TRIM("KWH/hh (per half hour)") AS DOUBLE
    ) IS NULL
    LIMIT 20
""").fetchdf()

print(non_numeric)

   consumption
0         Null
1         Null
2         Null
3         Null
4         Null
5         Null
6         Null
7         Null
8         Null
9         Null
10        Null
11        Null
12        Null
13        Null
14        Null
15        Null
16        Null
17        Null
18        Null
19        Null


In [47]:
# Check the 30 households
con.execute("""
    SELECT 
    LCLid,
        COUNT(*) AS readings,
        MIN(DateTime) AS first_reading,
        MAX(DateTime) AS last_reading
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY LCLid
    ORDER BY LCLid
""").fetchdf()

,LCLid,readings,first_reading,last_reading
0,MAC000002,24158,2012-10-12 00:30:00,2014-02-28 00:00:00
1,MAC000003,35469,2012-02-20 13:00:00,2014-02-28 00:00:00
2,MAC000004,31677,2012-05-08 13:00:00,2014-02-28 00:00:00
3,MAC000006,36461,2012-01-30 11:30:00,2014-02-28 00:00:00
4,MAC000007,25046,2012-09-24 12:00:00,2014-02-28 00:00:00
5,MAC000008,26013,2012-05-05 09:00:00,2013-10-30 00:00:00
6,MAC000009,25238,2012-09-20 10:00:00,2014-02-28 00:00:00
7,MAC000010,25049,2012-09-24 12:30:00,2014-02-28 00:00:00
8,MAC000011,23705,2012-10-22 11:00:00,2014-02-28 00:00:00
9,MAC000012,24670,2012-10-02 09:30:00,2014-02-28 00:00:00


In [48]:
con.execute(f"""
    SELECT COUNT(DISTINCT LCLid) AS unique_meters
    FROM read_csv_auto(
            '../data/raw/development/LCL-June2015v2_0.csv'
        )
""").df()

,unique_meters
0,30


In [49]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN LCLid IS NULL OR TRIM(LCLid) = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_lclid,

    SUM(
        CASE
            WHEN DateTime IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_datetime,

    SUM(
        CASE
            WHEN "KWH/hh (per half hour)" IS NULL
              OR TRIM("KWH/hh (per half hour)") = ''
            THEN 1 ELSE 0
        END
    ) AS invalid_consumption
FROM read_csv_auto(
    '../data/raw/development/LCL-June2015v2_0.csv'
)
""").fetchdf()

,total_rows,invalid_lclid,invalid_datetime,invalid_consumption
0,1000000,0.0,0.0,0.0


In [50]:
# Investigate duplicates
con.execute("""
SELECT
    COUNT(*) AS duplicate_groups,
    SUM(occurrences - 1) AS duplicate_rows
FROM (
    SELECT
        LCLid,
        DateTime,
        COUNT(*) AS occurrences
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY LCLid, DateTime
    HAVING COUNT(*) > 1
)
""").fetchdf()

,duplicate_groups,duplicate_rows
0,688,688.0


In [51]:
# Verify 30-minute frequency
con.execute("""
WITH readings AS (
    SELECT
        LCLid,
        DateTime,
        LEAD(DateTime) OVER (
            PARTITION BY LCLid
            ORDER BY DateTime
        ) AS next_datetime
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
)

SELECT
    next_datetime - DateTime AS interval,
    COUNT(*) AS occurrences
FROM readings
WHERE next_datetime IS NOT NULL
GROUP BY interval
ORDER BY occurrences DESC
""").fetchdf()

,interval,occurrences
0,0 days 00:30:00,999117
1,0 days 00:00:00,688
2,0 days 01:00:00,88
3,0 days 00:16:19,10
4,0 days 00:13:41,10
5,0 days 00:16:18,7
6,0 days 00:13:42,7
7,0 days 00:07:27,6
8,0 days 00:22:33,6
9,1 days 00:30:00,5


In [52]:
# investigate the duplicate timestamps
con.execute("""
SELECT *
FROM read_csv_auto(
    '../data/raw/development/LCL-June2015v2_0.csv'
)
WHERE LCLid = 'MAC000021'
  AND DateTime = '2014-02-28 00:00:00'
""").fetchdf()

,LCLid,stdorToU,DateTime,KWH/hh (per half hour)
0,MAC000021,Std,2014-02-28,0.737
1,MAC000021,Std,2014-02-28,0.737


In [53]:
con.execute("""
WITH duplicates AS (
    SELECT
        LCLid,
        DateTime
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY LCLid, DateTime
    HAVING COUNT(*) > 1
)
SELECT
    EXTRACT(YEAR FROM DateTime) AS year,
    EXTRACT(MONTH FROM DateTime) AS month,
    COUNT(*) AS duplicate_groups
FROM duplicates
GROUP BY year, month
ORDER BY year, month
""").fetchdf()

,year,month,duplicate_groups
0,2011,12,19
1,2012,1,19
2,2012,2,20
3,2012,3,21
4,2012,4,21
5,2012,5,23
6,2012,6,23
7,2012,7,24
8,2012,8,24
9,2012,9,24


In [54]:
con.execute("""
WITH duplicates AS (
    SELECT
        LCLid,
        DateTime,
        COUNT(*) AS occurrences
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY LCLid, DateTime
    HAVING COUNT(*) > 2
)
SELECT *
FROM duplicates
ORDER BY DateTime
LIMIT 100
""").fetchdf()

,LCLid,DateTime,occurrences


In [55]:
con.execute("""
WITH duplicates AS (
    SELECT
        LCLid,
        DateTime,
        COUNT(*) AS occurrences
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY LCLid, DateTime
    HAVING COUNT(*) > 2
)
SELECT *
FROM duplicates
ORDER BY DateTime
LIMIT 100
""").fetchdf()

,LCLid,DateTime,occurrences


In [56]:
duplicate_values = con.execute("""
SELECT
    LCLid,
    DateTime,
    COUNT(*) AS occurrences,
    COUNT(DISTINCT TRIM("KWH/hh (per half hour)")) AS distinct_values,
    MIN(TRIM("KWH/hh (per half hour)")) AS min_value,
    MAX(TRIM("KWH/hh (per half hour)")) AS max_value
FROM read_csv_auto(
    '../data/raw/development/LCL-June2015v2_0.csv'
)
GROUP BY LCLid, DateTime
HAVING COUNT(*) > 1
ORDER BY DateTime
""").fetchdf()

print(duplicate_values.head(30))

        LCLid   DateTime  occurrences  distinct_values min_value max_value
0   MAC000028 2011-12-15            2                1     0.066     0.066
1   MAC000027 2011-12-15            2                1     0.214     0.214
2   MAC000033 2011-12-15            2                1     0.265     0.265
3   MAC000035 2011-12-15            2                1     0.524     0.524
4   MAC000023 2011-12-15            2                1     0.175     0.175
5   MAC000026 2011-12-15            2                1     0.235     0.235
6   MAC000030 2011-12-15            2                1     0.049     0.049
7   MAC000019 2011-12-15            2                1     0.111     0.111
8   MAC000029 2011-12-15            2                1     0.059     0.059
9   MAC000021 2011-12-15            2                1     0.515     0.515
10  MAC000032 2011-12-15            2                1     0.062     0.062
11  MAC000020 2011-12-15            2                1     0.061     0.061
12  MAC000022 2011-12-15 

In [57]:
con.execute("""
WITH duplicate_groups AS (
    SELECT
        LCLid,
        DateTime,
        COUNT(*) AS occurrences,
        COUNT(DISTINCT TRIM("KWH/hh (per half hour)")) AS distinct_values
    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )
    GROUP BY LCLid, DateTime
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS duplicate_groups,
    SUM(CASE WHEN distinct_values = 1 THEN 1 ELSE 0 END)
        AS identical_duplicate_groups,
    SUM(CASE WHEN distinct_values > 1 THEN 1 ELSE 0 END)
        AS conflicting_duplicate_groups
FROM duplicate_groups
""").fetchdf()

,duplicate_groups,identical_duplicate_groups,conflicting_duplicate_groups
0,688,688.0,0.0


# RAW DATA
   │
   │ immutable
   ▼
DuckDB
   │
   ├── Validate
   │
   ├── Trim consumption
   │
   ├── Cast → DOUBLE
   │
   ├── Detect duplicates
   │
   └── Deduplicate identical observations
           │
           ▼
    PROCESSED DATA


    

for duplicate data check:
Duplicate timestamp
        ↓
Same household?
        ↓
Same timestamp?
        ↓
Same tariff?
        ↓
Same consumption?
        ↓
Same complete row?

##  Duplicate Investigation Conclusion

### Investigation Summary

The dataset was investigated for duplicate readings using the combination of:

- `LCLid`
- `DateTime`

The investigation found:

- **1,000,000 total rows** in the development dataset.
- **688 duplicate `(LCLid, DateTime)` groups** were identified.
- These groups resulted in **688 redundant rows**.
- Each duplicate group occurred exactly **twice**.
- All **688 duplicate groups had identical consumption values**.
- No conflicting consumption values were found among the duplicate groups.
- The duplicate records also showed consistent tariff values in the investigation.

### Data Quality Decision

The identified duplicates are treated as redundant observations rather than conflicting records.

Therefore, during the ETL stage:

1. The raw dataset will remain unchanged and immutable.
2. Duplicate observations will be detected using `(LCLid, DateTime)`.
3. Where duplicate records are identical, only one observation will be retained.
4. The deduplication will be performed in the processed dataset, not on the raw source.

### Engineering Decision

> The identified duplicate `(LCLid, DateTime)` groups contain identical observations based on the checks performed. These redundant records will be removed during deterministic ETL, while the raw dataset remains immutable.

###  Status

**Duplicate investigation: COMPLETE**

The dataset is ready to move to the deterministic ETL stage, where the documented cleaning decisions will be implemented.

In [58]:
duplicate_check = con.execute("""
    SELECT
        LCLid,
        DateTime,

        -- How many rows are in this group?
        COUNT(*) AS occurrences,

        -- How many different complete rows?
        COUNT(DISTINCT (
            LCLid,
            stdorToU,
            DateTime,
            "KWH/hh (per half hour)"
        )) AS distinct_complete_rows
        

    FROM read_csv_auto(
        '../data/raw/development/LCL-June2015v2_0.csv'
    )

    GROUP BY LCLid, DateTime

    HAVING COUNT(*) > 1
    AND COUNT(DISTINCT (
       LCLid,
       stdorToU,
       DateTime,
       "KWH/hh (per half hour)"
   )) > 1
""").fetchdf()



In [59]:
print(duplicate_check.head(20))

Empty DataFrame
Columns: [LCLid, DateTime, occurrences, distinct_complete_rows]
Index: []
